In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("Food Delivery Analytics") \
.getOrCreate()

orders_data = [
("O001","North","Delhi","Rest-01","Pizza","2024-02-01",450,35),
("O002","North","Delhi","Rest-01","Burger","2024-02-01",250,25),
("O003","North","Chandigarh","Rest-02","Pasta","2024-02-02",350,30),
("O004","South","Bangalore","Rest-03","Pizza","2024-02-01",500,40),
("O005","South","Chennai","Rest-04","Burger","2024-02-02",220,20),
("O006","South","Bangalore","Rest-03","Pasta","2024-02-03",380,32),
("O007","East","Kolkata","Rest-05","Pizza","2024-02-01",420,38),
("O008","East","Kolkata","Rest-05","Burger","2024-02-02",260,26),
("O009","East","Patna","Rest-06","Pasta","2024-02-03",300,28),
("O010","West","Mumbai","Rest-07","Pizza","2024-02-01",520,42),
("O011","West","Mumbai","Rest-07","Burger","2024-02-02",280,27),
("O012","West","Pune","Rest-08","Pasta","2024-02-03",340,31),
("O013","North","Delhi","Rest-01","Pizza","2024-02-04",480,37),
("O014","South","Chennai","Rest-04","Pizza","2024-02-04",510,41),
("O015","East","Patna","Rest-06","Burger","2024-02-04",240,24),
("O016","West","Pune","Rest-08","Pizza","2024-02-04",500,39),
("O017","North","Chandigarh","Rest-02","Burger","2024-02-05",260,26),
("O018","South","Bangalore","Rest-03","Burger","2024-02-05",290,29),
("O019","East","Kolkata","Rest-05","Pasta","2024-02-05",360,33),
("O020","West","Mumbai","Rest-07","Pasta","2024-02-05",390,34),
("O021","North","Delhi","Rest-01","Pasta","2024-02-06",370,30),
("O022","South","Chennai","Rest-04","Pasta","2024-02-06",330,29),
("O023","East","Patna","Rest-06","Pizza","2024-02-06",460,36),
("O024","West","Pune","Rest-08","Burger","2024-02-06",270,26)
]
columns = [
"order_id","region","city","restaurant_id",
"food_item","order_date","amount","delivery_time_min"
]
df_orders = spark.createDataFrame(orders_data, columns)
df_orders.show(5)
df_orders.printSchema()

+--------+------+----------+-------------+---------+----------+------+-----------------+
|order_id|region|      city|restaurant_id|food_item|order_date|amount|delivery_time_min|
+--------+------+----------+-------------+---------+----------+------+-----------------+
|    O001| North|     Delhi|      Rest-01|    Pizza|2024-02-01|   450|               35|
|    O002| North|     Delhi|      Rest-01|   Burger|2024-02-01|   250|               25|
|    O003| North|Chandigarh|      Rest-02|    Pasta|2024-02-02|   350|               30|
|    O004| South| Bangalore|      Rest-03|    Pizza|2024-02-01|   500|               40|
|    O005| South|   Chennai|      Rest-04|   Burger|2024-02-02|   220|               20|
+--------+------+----------+-------------+---------+----------+------+-----------------+
only showing top 5 rows
root
 |-- order_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- food_i

In [3]:
from pyspark.sql import functions as functions
from pyspark.sql import Window
df_orders.select("order_id","region","food_item","amount")

DataFrame[order_id: string, region: string, food_item: string, amount: bigint]

In [7]:
df_orders.withColumnRenamed("amount","order_value")

DataFrame[order_id: string, region: string, city: string, restaurant_id: string, food_item: string, order_date: string, order_value: bigint, delivery_time_min: bigint]

In [8]:
df_orders.withColumn("amount_in_hundreds",functions.col("amount")/100)

DataFrame[order_id: string, region: string, city: string, restaurant_id: string, food_item: string, order_date: string, amount: bigint, delivery_time_min: bigint, amount_in_hundreds: double]

In [9]:
df_orders.select("region","food_item").distinct()

DataFrame[region: string, food_item: string]

In [10]:
df_orders.select("order_id","order_date","region","city","food_item","amount")

DataFrame[order_id: string, order_date: string, region: string, city: string, food_item: string, amount: bigint]

In [12]:
df_orders.withColumn("order_day",functions.dayofmonth("order_date"))

DataFrame[order_id: string, region: string, city: string, restaurant_id: string, food_item: string, order_date: string, amount: bigint, delivery_time_min: bigint, order_day: int]

In [15]:
df_orders.filter(functions.col("amount") > 400).show()
df_orders.filter(functions.col("food_item") == "Pizza").show()
df_orders.filter(functions.col("city").isin("Delhi", "Mumbai")).show()

df_orders.filter(functions.col("delivery_time_min") > 35).show()

df_orders.filter((functions.col("amount") > 400) & (functions.col("city") == "Delhi")).show()

df_orders.filter(functions.col("amount") > 400).explain(True)

+--------+------+---------+-------------+---------+----------+------+-----------------+
|order_id|region|     city|restaurant_id|food_item|order_date|amount|delivery_time_min|
+--------+------+---------+-------------+---------+----------+------+-----------------+
|    O001| North|    Delhi|      Rest-01|    Pizza|2024-02-01|   450|               35|
|    O004| South|Bangalore|      Rest-03|    Pizza|2024-02-01|   500|               40|
|    O007|  East|  Kolkata|      Rest-05|    Pizza|2024-02-01|   420|               38|
|    O010|  West|   Mumbai|      Rest-07|    Pizza|2024-02-01|   520|               42|
|    O013| North|    Delhi|      Rest-01|    Pizza|2024-02-04|   480|               37|
|    O014| South|  Chennai|      Rest-04|    Pizza|2024-02-04|   510|               41|
|    O016|  West|     Pune|      Rest-08|    Pizza|2024-02-04|   500|               39|
|    O023|  East|    Patna|      Rest-06|    Pizza|2024-02-06|   460|               36|
+--------+------+---------+-----

In [17]:
pipeline_df=(
    df_orders
    .select("order_id","region","amount")
    .filter(functions.col("amount")>300)
    .withColumn("tax",functions.col("amount")*0.1)
)
pipeline_df.count()
pipeline_df.show()

+--------+------+------+----+
|order_id|region|amount| tax|
+--------+------+------+----+
|    O001| North|   450|45.0|
|    O003| North|   350|35.0|
|    O004| South|   500|50.0|
|    O006| South|   380|38.0|
|    O007|  East|   420|42.0|
|    O010|  West|   520|52.0|
|    O012|  West|   340|34.0|
|    O013| North|   480|48.0|
|    O014| South|   510|51.0|
|    O016|  West|   500|50.0|
|    O019|  East|   360|36.0|
|    O020|  West|   390|39.0|
|    O021| North|   370|37.0|
|    O022| South|   330|33.0|
|    O023|  East|   460|46.0|
+--------+------+------+----+



In [18]:
# 1. Check partitions
df_orders.rdd.getNumPartitions()

# 2. Repartition to 4
df4 = df_orders.repartition(4)

# 3. Coalesce to 1
df1 = df_orders.coalesce(1)

# 4. Write repartitioned
df4.write.mode("overwrite").parquet("output/repartitioned")

# 5. Write coalesced
df1.write.mode("overwrite").parquet("output/coalesced")

In [20]:
# 1. Total revenue per region
df_orders.groupBy("region").agg(functions.sum("amount").alias("total_revenue"))

# 2. Avg order amount per food item
df_orders.groupBy("food_item").agg(functions.avg("amount").alias("avg_amount"))

# 3. Max order per city
df_orders.groupBy("city").agg(functions.max("amount").alias("max_amount"))

# 4. Min delivery time per restaurant
df_orders.groupBy("restaurant_id").agg(functions.min("delivery_time_min").alias("min_delivery_time"))

# 5. Count orders per region
df_orders.groupBy("region").count()

# 6. Revenue per restaurant
df_orders.groupBy("restaurant_id").agg(functions.sum("amount").alias("total_revenue_restaurant"))

# 7. Region + food_item revenue
df_orders.groupBy("region", "food_item").agg(functions.sum("amount").alias("total_revenue"))

# 8. City-wise avg delivery time
df_orders.groupBy("city").agg(functions.avg("delivery_time_min").alias("avg_delivery_time"))

# 9. Revenue above threshold
df_orders.groupBy("region") \
    .agg(functions.sum("amount").alias("rev")) \
    .filter(functions.col("rev") > 5000)

# 10. Explain
df_orders.groupBy("region").sum("amount").explain(True)

== Parsed Logical Plan ==
'Aggregate ['region], ['region, unresolvedalias('sum(amount#6L))]
+- LogicalRDD [order_id#0, region#1, city#2, restaurant_id#3, food_item#4, order_date#5, amount#6L, delivery_time_min#7L], false

== Analyzed Logical Plan ==
region: string, sum(amount): bigint
Aggregate [region#1], [region#1, sum(amount#6L) AS sum(amount)#283L]
+- LogicalRDD [order_id#0, region#1, city#2, restaurant_id#3, food_item#4, order_date#5, amount#6L, delivery_time_min#7L], false

== Optimized Logical Plan ==
Aggregate [region#1], [region#1, sum(amount#6L) AS sum(amount)#283L]
+- Project [region#1, amount#6L]
   +- LogicalRDD [order_id#0, region#1, city#2, restaurant_id#3, food_item#4, order_date#5, amount#6L, delivery_time_min#7L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#1], functions=[sum(amount#6L)], output=[region#1, sum(amount)#283L])
   +- Exchange hashpartitioning(region#1, 200), ENSURE_REQUIREMENTS, [plan_id=238]
      +- Hash

In [22]:
w_region_date = Window.partitionBy("region").orderBy("order_date")
w_region_amt = Window.partitionBy("region").orderBy(functions.desc("amount"))
w_restaurant_time = Window.partitionBy("restaurant_id").orderBy("delivery_time_min")

# 1. Running total revenue
df_orders.withColumn("running_total",
    functions.sum("amount").over(w_region_date))

# 2. Rank orders by amount
df_orders.withColumn("rank", functions.rank().over(w_region_amt))

# 3. Row number per restaurant
df_orders.withColumn("row_num",
    functions.row_number().over(w_restaurant_time))

# 4. Dense rank food items by revenue
w_food = Window.partitionBy("region").orderBy(functions.desc("amount"))
df_orders.withColumn("dense_rank", functions.dense_rank().over(w_food))

# 5. Top 2 orders per region
df_orders.withColumn("rank", functions.rank().over(w_region_amt)) \
         .filter(functions.col("rank") <= 2)

# 6. Compare rank functions
df_orders.select(
    functions.rank().over(w_region_amt).alias("rank"),
    functions.dense_rank().over(w_region_amt).alias("dense_rank"),
    functions.row_number().over(w_region_amt).alias("row_number")
)

# 7. Cumulative delivery time
df_orders.withColumn("cum_delivery",
    functions.sum("delivery_time_min").over(w_restaurant_time))

DataFrame[order_id: string, region: string, city: string, restaurant_id: string, food_item: string, order_date: string, amount: bigint, delivery_time_min: bigint, cum_delivery: bigint]

In [24]:
df_orders.select("order_id", "amount").explain(True)

# Filter
df_orders.filter(functions.col("amount") > 400).explain(True)

# GroupBy
df_orders.groupBy("region").sum("amount").explain(True)

# Window
df_orders.withColumn("rank", functions.rank().over(w_region_amt)).explain(True)

== Parsed Logical Plan ==
'Project ['order_id, 'amount]
+- LogicalRDD [order_id#0, region#1, city#2, restaurant_id#3, food_item#4, order_date#5, amount#6L, delivery_time_min#7L], false

== Analyzed Logical Plan ==
order_id: string, amount: bigint
Project [order_id#0, amount#6L]
+- LogicalRDD [order_id#0, region#1, city#2, restaurant_id#3, food_item#4, order_date#5, amount#6L, delivery_time_min#7L], false

== Optimized Logical Plan ==
Project [order_id#0, amount#6L]
+- LogicalRDD [order_id#0, region#1, city#2, restaurant_id#3, food_item#4, order_date#5, amount#6L, delivery_time_min#7L], false

== Physical Plan ==
*(1) Project [order_id#0, amount#6L]
+- *(1) Scan ExistingRDD[order_id#0,region#1,city#2,restaurant_id#3,food_item#4,order_date#5,amount#6L,delivery_time_min#7L]

== Parsed Logical Plan ==
'Filter '`>`('amount, 400)
+- LogicalRDD [order_id#0, region#1, city#2, restaurant_id#3, food_item#4, order_date#5, amount#6L, delivery_time_min#7L], false

== Analyzed Logical Plan ==
order_